# Retail Data Cleaning

This notebook applies the project validation rules while preserving useful operational signals. Exact duplicate rows are removed; repeated purchases remain valid transactions. Returns and cancelled invoices are retained in a separate analysis subset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "rawretaildata.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(
    RAW_PATH,
    dtype={
        "Invoice": "string",
        "StockCode": "string",
        "Description": "string",
        "Customer ID": "string",
        "Country": "string",
    },
    parse_dates=["InvoiceDate"],
)

raw.columns = raw.columns.str.strip()
raw["Invoice"] = raw["Invoice"].str.strip()
raw["StockCode"] = raw["StockCode"].str.strip().str.upper()
raw["Description"] = raw["Description"].str.strip()
raw["Customer ID"] = raw["Customer ID"].str.strip()
raw["Quantity"] = pd.to_numeric(raw["Quantity"], errors="coerce")
raw["Price"] = pd.to_numeric(raw["Price"], errors="coerce")

raw.shape, raw.columns.tolist()

((1048575, 8),
 ['Invoice',
  'StockCode',
  'Description',
  'Quantity',
  'InvoiceDate',
  'Price',
  'Customer ID',
  'Country'])

In [3]:
# Build a lookup so missing descriptions can be recovered from the same StockCode.
description_lookup = (
    raw.loc[raw["Description"].notna() & raw["Description"].ne(""), ["StockCode", "Description"]]
    .drop_duplicates()
    .groupby("StockCode")["Description"]
    .first()
)

clean = raw.copy()
clean["Description"] = clean["Description"].replace("", pd.NA)
clean["Description"] = clean["Description"].fillna(clean["StockCode"].map(description_lookup))
clean["Description"] = clean["Description"].fillna("Unknown description")
clean["Customer ID"] = clean["Customer ID"].fillna("Guest")

clean["is_cancelled"] = clean["Invoice"].str.startswith("C", na=False)
clean["is_operational_code"] = clean["StockCode"].isin({"DOT", "POST", "M", "ADJUST"})
clean["is_exact_duplicate"] = clean.duplicated(keep="first")
clean["is_repeated_purchase"] = clean.duplicated(
    subset=["Invoice", "StockCode", "Description", "InvoiceDate", "Customer ID"],
    keep=False,
)
clean["is_outlier"] = clean["Quantity"].abs().gt(1000) | clean["Price"].abs().gt(1000)
clean["is_true_junk"] = (
    clean["Customer ID"].eq("Guest")
    & clean["Description"].eq("Unknown description")
    & clean["Price"].eq(0)
    & clean["Quantity"].lt(0)
)
clean["TotalPrice"] = clean["Quantity"] * clean["Price"]

# Remove exact duplicates only. Same-invoice repeated purchases remain valid records.
clean = clean.loc[~clean["is_exact_duplicate"]].copy()
clean = clean.loc[~clean["is_true_junk"] & ~clean["is_outlier"]].copy()

clean["is_partial_final_month"] = clean["InvoiceDate"].dt.to_period("M").eq(pd.Period("2011-12"))
clean["is_positive_sale"] = clean["Quantity"].gt(0) & ~clean["is_cancelled"]

# Returns require a negative quantity, cancelled invoice, known customer, and known description.
returns = clean.loc[
    clean["Quantity"].lt(0)
    & clean["is_cancelled"]
    & clean["Customer ID"].ne("Guest")
    & clean["Description"].ne("Unknown description")
].copy()

sales = clean.loc[clean["is_positive_sale"]].copy()
modeling_sales = sales.loc[~sales["is_operational_code"]].copy()
complete_month_sales = modeling_sales.loc[~modeling_sales["is_partial_final_month"]].copy()

clean.to_csv(INTERIM_DIR / "clean_transactions.csv", index=False)
returns.to_csv(INTERIM_DIR / "returns.csv", index=False)
modeling_sales.to_csv(INTERIM_DIR / "modeling_sales.csv", index=False)
complete_month_sales.to_csv(INTERIM_DIR / "complete_month_sales.csv", index=False)

clean.shape, returns.shape, modeling_sales.shape, complete_month_sales.shape

((1013408, 17), (18051, 17), (987719, 17), (981241, 17))

In [5]:
outlier_mask = raw["Quantity"].abs().gt(1000) | raw["Price"].abs().gt(1000)
true_junk_mask = (
    raw["Customer ID"].isna()
    & raw["Description"].isna()
    & raw["Price"].eq(0)
    & raw["Quantity"].lt(0)
)

audit = pd.Series(
    {
        "raw_rows": len(raw),
        "exact_duplicates_removed": int(raw.duplicated().sum()),
        "repeated_purchase_rows_retained": int(clean["is_repeated_purchase"].sum()),
        "guest_rows_after_tagging": int(clean["Customer ID"].eq("Guest").sum()),
        "operational_rows_flagged": int(clean["is_operational_code"].sum()),
        "outlier_rows_removed": int(outlier_mask.sum()),
        "true_junk_rows_removed": int(true_junk_mask.sum()),
        "clean_rows": len(clean),
        "return_rows": len(returns),
        "modeling_sales_rows": len(modeling_sales),
        "complete_month_sales_rows": len(complete_month_sales),
    }
)

assert not clean.duplicated().any()
assert not clean["is_outlier"].any()
assert not clean["is_true_junk"].any()
assert returns["Quantity"].lt(0).all() and returns["is_cancelled"].all()
assert returns["Customer ID"].ne("Guest").all()
assert returns["Description"].ne("Unknown description").all()
assert not complete_month_sales["is_partial_final_month"].any()

audit

raw_rows                           1048575
exact_duplicates_removed             34150
repeated_purchase_rows_retained      53552
guest_rows_after_tagging            228249
operational_rows_flagged              4714
outlier_rows_removed                   735
true_junk_rows_removed                2688
clean_rows                         1013408
return_rows                          18051
modeling_sales_rows                 987719
complete_month_sales_rows           981241
dtype: int64